In [1]:
### rag pipelines--- data ingestion to vector database
# 1. Data Ingestion: Load and preprocess data from various sources (e.g., CSV, PDFs, databases).
# 2. Text Processing: Clean and normalize text data (e.g., remove stop words, lemmatization).
# 3. Vectorization: Convert text data into vector representations using techniques like TF-IDF, Word2Vec, or BERT embeddings.
# 4. Indexing: Store vector representations in a vector database (e.g., FAISS, Pinecone) for efficient retrieval.
# 5. Retrieval: Implement a retrieval mechanism to fetch relevant documents based on user queries.
# 6. Response Generation: Use retrieved documents to generate responses, possibly leveraging language models for natural language generation.

In [1]:
import shutil
import os
import gc

# 1. Force Python to release any lingering objects
gc.collect()

db_path = "../data/vector_store"

# 2. Delete the directory entirely
if os.path.exists(db_path):
    try:
        shutil.rmtree(db_path)
        print(f"✅ Successfully deleted locked database at {db_path}")
    except Exception as e:
        print(f"❌ Could not delete directory. You may need to manually delete the folder {db_path} in your file explorer: {e}")
else:
    print("Database folder does not exist. Ready to start fresh.")

✅ Successfully deleted locked database at ../data/vector_store


In [2]:
# data ingestion to vector database

In [3]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [4]:
import os
from pathlib import Path

In [5]:
def process_all_pdf(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files in {pdf_directory}")

    for pdf_file in pdf_files:
        print(f"Processing {pdf_file.name}...")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata["source"] = str(pdf_file)  # ✅ fix
                all_documents.append(doc)               # ✅ add

        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")

    return all_documents  # ✅ missing return

In [6]:
all_pdf_documents = process_all_pdf("../data/pdf")

Found 1 PDF files in ../data/pdf
Processing [O_G_Kakde]_Algorithms_for_compiler_design(BookZZ.org) (2).pdf...


In [7]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    chunks = text_splitter.split_documents(documents)
    print(f"Created {len(chunks)} chunks from {len(documents)} documents.")

    if chunks:
        print(f"First chunk metadata: {chunks[0].metadata}")

    return chunks

In [8]:
chunks = split_documents(all_pdf_documents)

Created 587 chunks from 355 documents.
First chunk metadata: {'producer': 'wPDF - http://www.wptools.de', 'creator': 'wPDF by wpCubed GmbH', 'creationdate': '2008-12-07T02:20:23+00:00', 'source': '../data/pdf/[O_G_Kakde]_Algorithms_for_compiler_design(BookZZ.org) (2).pdf', 'file_path': '../data/pdf/[O_G_Kakde]_Algorithms_for_compiler_design(BookZZ.org) (2).pdf', 'total_pages': 355, 'format': 'PDF 1.6', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2017-03-24T15:55:40+05:30', 'trapped': '', 'modDate': "D:20170324155540+05'30'", 'creationDate': 'D:20081207022023Z', 'page': 0}


In [9]:
chunks

[Document(metadata={'producer': 'wPDF - http://www.wptools.de', 'creator': 'wPDF by wpCubed GmbH', 'creationdate': '2008-12-07T02:20:23+00:00', 'source': '../data/pdf/[O_G_Kakde]_Algorithms_for_compiler_design(BookZZ.org) (2).pdf', 'file_path': '../data/pdf/[O_G_Kakde]_Algorithms_for_compiler_design(BookZZ.org) (2).pdf', 'total_pages': 355, 'format': 'PDF 1.6', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2017-03-24T15:55:40+05:30', 'trapped': '', 'modDate': "D:20170324155540+05'30'", 'creationDate': 'D:20081207022023Z', 'page': 0}, page_content='Algorithms for Compiler Design\nby O.G. Kakde \nISBN:1584501006\nCharles River Media © 2002 (334 pages)\nThis text teaches the fundamental algorithms that underlie modern compilers, and focuses on the\n"front-end" of compiler design--lexical analysis, parsing, and syntax.\nTable of Contents\nAlgorithms for Compiler Design\nPreface\nChapter 1\n- Introduction\nChapter 2\n- Finite Automata and Regular Expressions\nChapter

In [10]:
 #embedding and vector database storage will be the next steps

In [11]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid 
from typing import List, Dict, Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [12]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


/var/folders/93/t_zj5n015339_5q_qlth41340000gn/T/ipykernel_10029/540119965.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [44]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection with Cosine Similarity
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "PDF document embeddings for RAG",
                    "hnsw:space": "cosine"  # ✅ Explicitly set to cosine distance
                }
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

# Initialize the vector store
vectorstore = VectorStore()


Vector store initialized. Collection: pdf_documents
Existing documents in collection: 587


In [14]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 587 texts...


Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Generated embeddings with shape: (587, 384)
Adding 587 documents to vector store...
Successfully added 587 documents to vector store
Total documents in collection: 587


In [82]:
# retrieval pipeline from vector database

In [15]:
class RAGRetriever:
    """Handles retrieval of documents from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, n_results: int = 3, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query string
            n_results: Number of documents to return
            score_threshold: Minimum similarity score (0.0 to 1.0)
            
        Returns:
            List of dictionaries containing document content, metadata, and scores
        """
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search collection
        results = self.vector_store.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=n_results
        )
        
        retrieved_docs = []
        
        # Process results (Chroma returns lists of lists)
        if results['documents'] and results['documents'][0]:
            for i in range(len(results['documents'][0])):
                distance = results['distances'][0][i]
                
                # Since we are using Cosine Similarity:
                # Similarity = 1 - distance
                similarity_score = 1 - distance
                
                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        'content': results['documents'][0][i],
                        'metadata': results['metadatas'][0][i],
                        'similarity_score': similarity_score
                    })
        
        return retrieved_docs

In [19]:
# Re-initialize after updating the class
rag_retriever = RAGRetriever(vectorstore, embedding_manager)

# Now your test should work perfectly
query = "compiler "
results = rag_retriever.retrieve(query, n_results=3, score_threshold=0.3)

Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)


In [20]:
results

[{'content': 'chapters on code generation and optimization complete a solid foundation for learning the broader requirements of an\nentire compiler design.\nFEATURES\nFocuses on the “front-end” of compiler design—lexical analysis, parsing, and syntax—topics basic to any\nintroduction to compiler design\nCovers storage management, error handling, and recovery\nIntroduces important “back-end” programming concepts, including code generation and optimization',
  'metadata': {'modDate': "D:20170324155540+05'30'",
   'doc_index': 3,
   'creationdate': '2008-12-07T02:20:23+00:00',
   'page': 1,
   'creationDate': 'D:20081207022023Z',
   'moddate': '2017-03-24T15:55:40+05:30',
   'title': '',
   'content_length': 438,
   'author': '',
   'format': 'PDF 1.6',
   'subject': '',
   'producer': 'wPDF - http://www.wptools.de',
   'keywords': '',
   'total_pages': 355,
   'file_path': '../data/pdf/[O_G_Kakde]_Algorithms_for_compiler_design(BookZZ.org) (2).pdf',
   'trapped': '',
   'source': '../dat

In [31]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_core.messages import HumanMessage



### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="openai/gpt-oss-120b",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,n_results=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    response = llm.invoke([
    HumanMessage(content=prompt.format(context=context, query=query))])
    return response.content

In [34]:
answer=rag_simple("what is syntactical analysis?", rag_retriever, llm)
print("Answer:", answer)

Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Answer: **Syntactic analysis** (or syntax analysis) is the compiler phase that takes the stream of tokens produced by the lexical analyzer and checks whether the token sequence conforms to the language’s grammar. It groups tokens according to the grammatical rules, builds a parse tree (top‑down or bottom‑up), and reports any violations as syntax errors.


In [35]:
# enhanced pipeline

In [37]:
from langchain_core.messages import HumanMessage

def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    
    results = retriever.retrieve(query, n_results=top_k, score_threshold=min_score)

    if not results:
        return {
            'answer': 'No relevant context found.',
            'sources': [],
            'confidence': 0.0,
            'context': ''
        }
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.

Context:
{context}

Question: {query}

Answer:"""
    
    response = llm.invoke([
        HumanMessage(content=prompt)
    ])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    
    if return_context:
        output['context'] = context
        
    return output
# Example usage:
result = rag_advanced("why is compiler important? why should i stduy it", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Answer: A compiler is the bridge that turns the human‑readable code you write into the machine‑level instructions a computer can actually execute. Because every program that runs on a computer must be compiled (or interpreted) into low‑level code, compilers are **fundamental to all software systems**.  

Studying compilers matters because:

* **Understanding the translation process** – you learn how high‑level constructs (loops, functions, objects) are broken down into efficient machine operations.  
* **Improving program quality** – knowledge of lexical analysis, parsing, syntax, storage management, and error recovery lets you write code that compiles cleanly and runs faster.  
* **Performance optimization** – the back‑end topics of code generation and optimization teach techniques to make programs run with less time and memory.  
* **Building tools and languages** – the algorithms you master are the building blocks for creating new languages,

In [38]:
from typing import List, Dict, Any
from langchain_core.messages import HumanMessage, SystemMessage
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        
        # ✅ FIX 1
        results = self.retriever.retrieve(question, n_results=top_k, score_threshold=min_score)

        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])

            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]

            prompt = f"""Use the following context to answer the question concisely.

Context:
{context}

Question: {question}

Answer:"""

            # Optional streaming (simulation only)
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()

            # ✅ FIX 2
            response = self.llm.invoke([
                SystemMessage(content="You are a helpful and concise assistant."),
                HumanMessage(content=prompt)
            ])

            answer = response.content

        # Citations
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Summarization
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"

            # ✅ FIX 3
            summary_resp = self.llm.invoke([
                HumanMessage(content=summary_prompt)
            ])

            summary = summary_resp.content

        # Store history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

In [41]:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is comipler design", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Streaming answer:
Use the following context to answer the question concisely.

Context:
O.G. Kakde. Algorithms for Compiler Design.
Original ISBN: 81-7008-100-6
All brand names and product names mentioned in this book are trademarks or service marks of their
respective companies. Any omission or misuse (of any kind) of service marks or trademarks should not be
regarded as intent to infringe on the property of others. The publisher recognizes and respects all marks used
by companies, manufacturers, and developers as a means to distinguish their products.
02 7 6 5 4 3 2 First Edition
CHARLES RIVER MEDIA titles are available for site license or bulk purchase by institutions, user groups,
corporations, etc. For additional information, please contact the Special Sales Department at 781-740-0400.
Acknowledgments
The author wishes to thank all of the colleagues in the Department of Electronics and Computer Science
Engineering at Visvesvaraya Regional 